# Каталог статей arXiv по информационному поиску

## Импорты

In [1]:
import os
import time
import xml.etree.ElementTree as ET
from contextlib import closing
from datetime import datetime

import pandas as pd
import psycopg2
import requests
from dotenv import load_dotenv
from loguru import logger
from psycopg2.extras import execute_values

## Конфигурация

In [2]:
load_dotenv()
query = "cat:cs.IR AND submittedDate:[202510010000 TO 202510312359]"
db = {
    "host": os.environ["POSTGRES_HOST"],
    "port": os.environ["POSTGRES_PORT"],
    "dbname": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
    "connect_timeout": 10,
}

NS = {
    "a": "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
    "os": "http://a9.com/-/spec/opensearch/1.1/",
}

In [3]:
def connect():
    try:
        return psycopg2.connect(**db)
    except psycopg2.OperationalError as exc:
        raise RuntimeError("Не удалось подключиться к PostgreSQL. Проверьте Podman и .env.") from exc

## Extract

In [4]:
def fetch_entries(query):
    entries = []
    while True:
        params = {"search_query": query, "start": len(entries), "max_results": 100,
                  "sortBy": "submittedDate", "sortOrder": "ascending"}
        try:
            response = requests.get("https://export.arxiv.org/api/query", params=params, timeout=60)
            response.raise_for_status()
            feed = ET.fromstring(response.content)
        except (requests.RequestException, ET.ParseError) as exc:
            raise RuntimeError(f"Не удалось прочитать arXiv со смещения {len(entries)}") from exc

        total = int(feed.findtext("os:totalResults", namespaces=NS))
        page = feed.findall("a:entry", NS)
        entries.extend(page)
        logger.info("Получено статей: {}/{}", len(entries), total)
        if not page or len(entries) >= total:
            return entries, total
        time.sleep(3.1)


entries, total = fetch_entries(query)

2026-11-22 15:07:47.725 +03:00 | INFO     | __main__:fetch_entries:16 - Получено статей: 100/384


2026-11-22 15:07:51.197 +03:00 | INFO     | __main__:fetch_entries:16 - Получено статей: 200/384


2026-11-22 15:07:54.882 +03:00 | INFO     | __main__:fetch_entries:16 - Получено статей: 300/384


2026-11-22 15:07:58.492 +03:00 | INFO     | __main__:fetch_entries:16 - Получено статей: 384/384


## Transform

In [5]:
def parse_paper(entry):
    def text(field):
        return " ".join(entry.findtext(f"a:{field}", namespaces=NS).split())

    arxiv_id, version = text("id").rsplit("/", 1)[-1].rsplit("v", 1)
    return (
        arxiv_id, int(version), text("title"), text("summary"),
        [author.text.strip() for author in entry.findall("a:author/a:name", NS)],
        entry.find("arxiv:primary_category", NS).attrib["term"],
        [category.attrib["term"] for category in entry.findall("a:category", NS)],
        datetime.fromisoformat(text("published")), datetime.fromisoformat(text("updated")),
        entry.find("a:link[@rel='alternate']", NS).attrib["href"],
    )


papers = [parse_paper(entry) for entry in entries]
source_ids = {paper[0] for paper in papers}
if len(papers) != total or len(source_ids) != total:
    raise ValueError(f"Ожидалось {total} уникальных статей из arXiv.")

## Load

In [6]:
upsert = """
    INSERT INTO stg.arxiv_papers AS p
        (arxiv_id, version, title, abstract, authors, primary_category, categories,
         published_at, updated_at, article_url)
    VALUES %s
    ON CONFLICT (arxiv_id) DO UPDATE SET
        (version, title, abstract, authors, primary_category, categories, published_at, updated_at, article_url) =
        (EXCLUDED.version, EXCLUDED.title, EXCLUDED.abstract, EXCLUDED.authors, EXCLUDED.primary_category,
         EXCLUDED.categories, EXCLUDED.published_at, EXCLUDED.updated_at, EXCLUDED.article_url)
    WHERE EXCLUDED.version > p.version
    RETURNING arxiv_id
"""

In [7]:
def load_papers(papers):
    with closing(connect()) as conn, conn.cursor() as cur:
        saved = execute_values(cur, upsert, papers, fetch=True)
        conn.commit()
    logger.info("Записано: {} из {}", len(saved), len(papers))
    return len(saved)


written = load_papers(papers)

2026-11-22 15:07:58.641 +03:00 | INFO     | __main__:load_papers:5 - Записано: 0 из 384


## Проверка

In [8]:
with closing(connect()) as conn, conn.cursor() as cur:
    cur.execute("SELECT COUNT(*), COUNT(DISTINCT arxiv_id) FROM stg.arxiv_papers")
    rows, unique_ids = cur.fetchone()
    cur.execute("""
        SELECT arxiv_id, version, title, primary_category, published_at
        FROM stg.arxiv_papers ORDER BY arxiv_id LIMIT 5
    """)
    preview = pd.DataFrame(cur.fetchall(), columns=[col.name for col in cur.description])

if rows != total:
    raise ValueError(f"В PostgreSQL {rows} строк, в arXiv {total}")
print(f"Строк: {rows} | Уникальных ID: {unique_ids} | Вставлено или обновлено: {written}")
preview

Строк: 384 | Уникальных ID: 384 | Вставлено или обновлено: 0


,arxiv_id,version,title,primary_category,published_at
0,2510.00671,2,Milco: Learned Sparse Retrieval Across Languag...,cs.IR,2025-10-01 08:58:25+00:00
1,2510.00694,1,ALARB: An Arabic Legal Argument Reasoning Benc...,cs.CL,2025-10-01 09:15:41+00:00
2,2510.00706,1,AttentionDep: Domain-Aware Attention for Expla...,cs.AI,2025-10-01 09:20:53+00:00
3,2510.00861,2,Erase to Improve: Erasable Reinforcement Learn...,cs.CL,2025-10-01 13:10:36+00:00
4,2510.00887,2,On Listwise Reranking for Corpus Feedback,cs.IR,2025-10-01 13:34:02+00:00
